# 4.3 — Build Automated Document Processing Pipelines

**Exam domain:** Domain 4.0 — Document Processing · **Weight:** 15%

## The problem this solves

Running the extraction query by hand works until it doesn't. Someone is on holiday, a batch arrives
at 2 a.m., or the same file gets processed twice because nobody remembers what was done yesterday.
What you want is for a document landing on a stage to become a row in a table, without a person in
the loop — and, when it doesn't, for something to say so.

Snowflake builds that out of two objects you may already know from ordinary data engineering: a
**stream** that remembers what is new, and a **task** that runs when there is something to do. The
document functions are just the payload in the middle.

## What you will be able to do

- Create a stream on a stage directory table and read what it captured
- Wrap parsing and extraction in a stored procedure that processes only new files
- Schedule that procedure with a task that skips its run when there is nothing waiting
- Chain dependent tasks and inspect the resulting graph
- Debug a pipeline that ran, reported success, and produced no rows

## Before you start

- Finish 4.1 and 4.2 — this notebook assumes `AI_EXTRACT` accessors and a working stage
- Run `setup/dataset.sql`, upload the `sample_docs/` files and refresh the stage
- Your role needs `CREATE TABLE`, `CREATE STREAM`, `CREATE TASK` and `CREATE PROCEDURE` in
  `GENAI_STUDY.PUBLIC`, plus `EXECUTE TASK` on the account and `USAGE` on a warehouse

📖 **Snowflake documentation for this notebook**
- [Introduction to streams](https://docs.snowflake.com/en/user-guide/streams-intro)
- [CREATE STREAM](https://docs.snowflake.com/en/sql-reference/sql/create-stream)
- [CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)
- [Task graphs](https://docs.snowflake.com/en/user-guide/tasks-graphs)
- [SYSTEM$STREAM_HAS_DATA](https://docs.snowflake.com/en/sql-reference/functions/system_stream_has_data)
- [AI_EXTRACT](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)


---
## The shape of the pipeline

```
document lands on the stage
        │
        ▼  (ALTER STAGE ... REFRESH, or AUTO_REFRESH notification)
directory table updates
        │
        ▼
STREAM on the stage  ──  remembers which files are new since it was last consumed
        │
        ▼
TASK  ──  wakes on a schedule, skips the run when the stream is empty
        │
        ▼
stored procedure: AI_EXTRACT (and AI_PARSE_DOCUMENT if you want the text)
        │
        ▼
structured table
```

A **stream** is a bookmark, not a copy. It records the change data between an offset and the current
state of the object it watches, and you read it like a table. Streams on directory tables are
**standard streams** only — no append-only or insert-only variant here.

| Metadata column | What it tells you |
|---|---|
| `METADATA$ACTION` | The recorded DML operation: `INSERT` or `DELETE` |
| `METADATA$ISUPDATE` | Whether the operation was part of an `UPDATE` |
| `METADATA$ROW_ID` | A unique, immutable row id for tracking a row over time |

The rule that catches everyone: **a stream advances its offset only when it is used in a DML
transaction** — an `INSERT … SELECT FROM stream`, a CTAS, or a `COPY INTO <location>`. A plain
`SELECT` shows you the rows and changes nothing. That is convenient for debugging and lethal if two
different jobs both consume the same stream: the first one to run a DML wins and the second sees an
empty stream.

→ [More on streams](https://docs.snowflake.com/en/user-guide/streams-intro)

A **task** is a scheduled unit of SQL. It can run on a `SCHEDULE`, or `AFTER` another task as part
of a graph. A `WHEN` clause gates the run, and `SYSTEM$STREAM_HAS_DATA('<stream>')` is the standard
gate for this pattern — the run is skipped, and the warehouse is not started, when there is nothing
to do.

→ [More on tasks](https://docs.snowflake.com/en/sql-reference/sql/create-task)


In [ ]:
%%sql
-- Step 1: the target table for structured results.
CREATE OR REPLACE TABLE GENAI_STUDY.PUBLIC.PARSED_DOCUMENTS (
    doc_id          NUMBER AUTOINCREMENT PRIMARY KEY,
    filename        VARCHAR(500),
    file_size_bytes NUMBER,
    parse_mode      VARCHAR(10),
    raw_text        VARCHAR(16000),
    invoice_number  VARCHAR(100),
    invoice_date    DATE,
    total_amount    FLOAT,
    vendor_name     VARCHAR(200),
    parse_status    VARCHAR(20) DEFAULT 'pending',
    parsed_at       TIMESTAMP_NTZ,
    error_message   VARCHAR(1000)
);

-- Step 2: a stream on the stage directory table.
-- The stage must have DIRECTORY = (ENABLE = TRUE), and the directory metadata must be
-- refreshed before the stream can see anything:  ALTER STAGE ... REFRESH;
CREATE OR REPLACE STREAM GENAI_STUDY.PUBLIC.DOCS_STAGE_STREAM
    ON STAGE GENAI_STUDY.PUBLIC.DOCS_STAGE
    COMMENT = 'Captures new document uploads for automated processing';


In [ ]:
%%sql -r pipeline_objects_2
SHOW STREAMS LIKE 'DOCS_STAGE_STREAM' IN SCHEMA GENAI_STUDY.PUBLIC;


### Why a stored procedure and not just a task body

A task can run a single SQL statement. The moment the pipeline needs a multi-step transaction,
error branching or a return value you can read in the run history, it needs a procedure.

The important design decision inside it is **how many AI calls you make per document**. Reading the
FILE once with `AI_EXTRACT` gives you the fields. Adding `AI_PARSE_DOCUMENT` gives you the text as
well, at the price of a second pass over the same pages. The procedure below does both deliberately,
because the table has a `raw_text` column that later steps read — if yours does not, drop the parse
and halve the bill.

A procedure runs with **owner's rights** by default: it executes as the role that owns it, not the
role that called it. That role needs the AI privileges and stage access, or the procedure fails for
every caller no matter how well privileged the caller is.

→ [More on AI_EXTRACT return shapes](https://docs.snowflake.com/en/sql-reference/functions/ai_extract)


In [ ]:
%%sql
-- ============================================================
-- Step 3: the procedure that processes new documents.
-- One AI_EXTRACT call per document, reading the FILE directly, plus one
-- AI_PARSE_DOCUMENT because this table keeps the text. Both use the error-
-- returning form so a bad file lands as a row with a message, not a silent NULL.
-- ============================================================
CREATE OR REPLACE PROCEDURE GENAI_STUDY.PUBLIC.PROCESS_NEW_DOCUMENTS()
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
BEGIN
    INSERT INTO GENAI_STUDY.PUBLIC.PARSED_DOCUMENTS
        (filename, file_size_bytes, parse_mode, raw_text,
         invoice_number, invoice_date, total_amount, vendor_name,
         parse_status, parsed_at, error_message)
    WITH new_files AS (
        -- Reading the stream inside an INSERT is what advances its offset.
        SELECT RELATIVE_PATH, SIZE
        FROM GENAI_STUDY.PUBLIC.DOCS_STAGE_STREAM
        WHERE METADATA$ACTION = 'INSERT'
          AND LOWER(RELATIVE_PATH) LIKE '%.pdf'
          AND SIZE < 104857600
    ),
    processed AS (
        SELECT
            f.RELATIVE_PATH,
            f.SIZE,
            AI_PARSE_DOCUMENT(
                TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', f.RELATIVE_PATH),
                {'mode': 'LAYOUT'}, TRUE
            ) AS parsed,                       -- {value, error, metadata}
            AI_EXTRACT(
                file           => TO_FILE('@GENAI_STUDY.PUBLIC.DOCS_STAGE', f.RELATIVE_PATH),
                responseFormat => {
                    'invoice_number': 'What is the invoice reference number?',
                    'invoice_date':   'What is the invoice date in YYYY-MM-DD format?',
                    'total_amount':   'What is the total amount due, as a number?',
                    'vendor_name':    'What company issued this invoice?'
                }
            ) AS fields                        -- {error, response}
        FROM new_files f
    )
    SELECT
        RELATIVE_PATH,
        SIZE,
        'LAYOUT',
        parsed:value:content::VARCHAR,
        fields:response:invoice_number::VARCHAR,
        TRY_TO_DATE(fields:response:invoice_date::VARCHAR),
        TRY_TO_DOUBLE(fields:response:total_amount::VARCHAR),
        fields:response:vendor_name::VARCHAR,
        IFF(fields:error IS NULL AND parsed:error IS NULL, 'success', 'failed'),
        CURRENT_TIMESTAMP(),
        COALESCE(fields:error::VARCHAR, parsed:error::VARCHAR)
    FROM processed;

    RETURN 'Processed ' || SQLROWCOUNT || ' documents';
END;
$$;


In [ ]:
%%sql -r proc_definition_2
-- A stored procedure runs with owner's rights: the OWNER role needs USE AI FUNCTIONS,
-- one of the Cortex database roles, and READ on the stage -- not the caller.
-- Expect the sample invoices (KF-2041, AV-8817, TS-5530) plus the contract and the
-- financial statement to come through on the first run.

CALL GENAI_STUDY.PUBLIC.PROCESS_NEW_DOCUMENTS();


> ### ⚠️ Common misconceptions
>
> **"I created the task, so it is running."**
> Newly created and cloned tasks are created **suspended**. Until `ALTER TASK … RESUME` runs, the
> schedule is inert and the run history is empty — not failing, simply absent. This is the single
> most common cause of a pipeline that "silently does nothing".
> → [CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)
>
> **"I checked the stream with a SELECT, so it's consumed now."**
> A stream advances its offset only when it is used in a **DML transaction** — `INSERT … SELECT`, a
> CTAS, or `COPY INTO <location>`. A bare `SELECT` is free to run as often as you like. The flip
> side: if two jobs read the same stream and either performs DML, the other one's rows vanish.
> → [Streams](https://docs.snowflake.com/en/user-guide/streams-intro)
>
> **"`SYSTEM$STREAM_HAS_DATA` returned TRUE, so there is definitely work."**
> It is documented to avoid false negatives but is *not* guaranteed to avoid false positives. If
> your task's `WHEN` gate fires and the body does no DML on the stream, the function keeps returning
> TRUE and the task keeps starting a warehouse for nothing.
> → [SYSTEM$STREAM_HAS_DATA](https://docs.snowflake.com/en/sql-reference/functions/system_stream_has_data)
>
> **"A stream on a stage sees files as soon as they are uploaded."**
> It sees what the directory table sees. Files that arrive outside Snowflake require
> `ALTER STAGE … REFRESH` (or `AUTO_REFRESH`) before the stream has anything to record.
> → [CREATE STREAM](https://docs.snowflake.com/en/sql-reference/sql/create-stream)


### Scheduling it

```sql
SCHEDULE = '5 MINUTE'                       -- interval form
SCHEDULE = 'USING CRON 0 2 * * * UTC'       -- cron form
```

Give the task a `WAREHOUSE` and it runs on your compute. Omit `WAREHOUSE` and it is serverless,
sized from `USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE`; a serverless task also needs the
`EXECUTE MANAGED TASK` account privilege, and a triggered task needs `TARGET_COMPLETION_INTERVAL`.

The `WHEN SYSTEM$STREAM_HAS_DATA(...)` gate is what makes a five-minute schedule affordable: the
run is skipped rather than started when the stream is empty, so the warehouse never wakes.

**Privileges the task needs, held by its owner:** `CREATE TASK` on the schema to make it,
`EXECUTE TASK` on the account to run it, `USAGE` on the warehouse, and everything the body needs —
the AI grants and stage access from 4.2.


In [ ]:
%%sql
-- Step 4: Create a TASK to run the procedure automatically
-- Triggers when the stage stream has new data
CREATE OR REPLACE TASK GENAI_STUDY.PUBLIC.AUTO_PARSE_DOCUMENTS
    WAREHOUSE   = COMPUTE_WH
    SCHEDULE    = '5 MINUTE'
    WHEN SYSTEM$STREAM_HAS_DATA('GENAI_STUDY.PUBLIC.DOCS_STAGE_STREAM')
AS
    CALL GENAI_STUDY.PUBLIC.PROCESS_NEW_DOCUMENTS();

-- Resume the task (suspended by default)
ALTER TASK GENAI_STUDY.PUBLIC.AUTO_PARSE_DOCUMENTS RESUME;


In [ ]:
%%sql -r task_setup_2
-- Check task status and last run
SHOW TASKS LIKE 'AUTO_PARSE_DOCUMENTS' IN SCHEMA GENAI_STUDY.PUBLIC;


In [ ]:
%%sql -r task_setup_3
-- Task run history
SELECT
    NAME,
    STATE,
    SCHEDULED_TIME,
    COMPLETED_TIME,
    RETURN_VALUE,
    ERROR_MESSAGE
FROM TABLE(
    INFORMATION_SCHEMA.TASK_HISTORY(
        SCHEDULED_TIME_RANGE_START => DATEADD('hour', -24, CURRENT_TIMESTAMP()),
        TASK_NAME => 'AUTO_PARSE_DOCUMENTS'
    )
)
ORDER BY SCHEDULED_TIME DESC;


### Chaining tasks into a graph

`AFTER <predecessor>` makes a task a child. A child runs after all of its predecessors finish
successfully, which gives you ordering without a scheduler.

Two rules about resuming that decide whether a graph ever runs:

- Resuming the root does **not** resume its children. Either resume each child with
  `ALTER TASK … RESUME`, or call `SYSTEM$TASK_DEPENDENTS_ENABLE` on the root to resume the whole
  graph at once.
- Suspending the root leaves the children in whatever state they were in; you do not have to suspend
  them individually, and future scheduled runs of the root are cancelled while a run already in
  progress completes.

A graph is limited to 1,000 tasks, and a single task to 100 parents and 100 children.

The trade-off in splitting work across tasks: each boundary gives you a retry point and a separate
line in the run history, and costs you a warehouse resume and a little latency. Two tasks for two
genuinely separate failure modes is good design; six tasks because the SQL felt long is not.

→ [More on task graphs](https://docs.snowflake.com/en/user-guide/tasks-graphs)


In [ ]:
%%sql
-- Chain a dependent task with AFTER: fill in anything the first pass left NULL,
-- working from the stored text rather than re-reading the PDF.
CREATE OR REPLACE TASK GENAI_STUDY.PUBLIC.ENRICH_PARSED_DOCS
    WAREHOUSE = COMPUTE_WH
    AFTER GENAI_STUDY.PUBLIC.AUTO_PARSE_DOCUMENTS
AS
UPDATE GENAI_STUDY.PUBLIC.PARSED_DOCUMENTS
SET
    vendor_name  = AI_EXTRACT(raw_text,
                     {'vendor_name': 'What is the vendor company name?'}):response:vendor_name::VARCHAR,
    total_amount = TRY_TO_DOUBLE(
                     AI_EXTRACT(raw_text,
                       {'total': 'What is the total dollar amount due?'}):response:total::VARCHAR)
WHERE parse_status = 'success'
  AND vendor_name IS NULL
  AND raw_text IS NOT NULL;

-- Resuming the root does NOT resume its children. Resume each child yourself, or call
-- SYSTEM$TASK_DEPENDENTS_ENABLE on the root to enable the whole graph in one go.
ALTER TASK GENAI_STUDY.PUBLIC.ENRICH_PARSED_DOCS RESUME;
ALTER TASK GENAI_STUDY.PUBLIC.AUTO_PARSE_DOCUMENTS RESUME;


In [ ]:
%%sql -r task_dag_2
-- SHOW TASKS is where the graph structure lives: its output includes the predecessors,
-- the schedule, the WHEN condition and the state. INFORMATION_SCHEMA.TASK_HISTORY()
-- reports runs, not structure, and has no predecessors column.
SHOW TASKS IN SCHEMA GENAI_STUDY.PUBLIC;


In [ ]:
%%sql -r task_dag_3
SELECT "name", "state", "schedule", "predecessors", "condition"
FROM TABLE(RESULT_SCAN(LAST_QUERY_ID()));


In [ ]:
%%sql -r task_dag_4
-- Dependency tree from the root:
SELECT *
FROM TABLE(INFORMATION_SCHEMA.TASK_DEPENDENTS(
    TASK_NAME => 'GENAI_STUDY.PUBLIC.AUTO_PARSE_DOCUMENTS',
    RECURSIVE => TRUE
));


In [ ]:
%%sql -r task_dag_5
-- Run history (this is where SCHEDULED_TIME / STATE / ERROR_MESSAGE live):
SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, RETURN_VALUE, ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD('hour', -24, CURRENT_TIMESTAMP())
))
ORDER BY SCHEDULED_TIME DESC;


> ### 🤔 Stop and think
>
> - A five-minute schedule gated by `SYSTEM$STREAM_HAS_DATA` costs almost nothing when idle, but
>   means a document can sit for five minutes. What would you have to know about the business before
>   choosing one minute, five, or nightly — and who pays for the difference?
> - The procedure here makes two AI calls per document so the table can keep `raw_text`. If nobody
>   has queried `raw_text` in six months, is it a cache or a liability? How would you find out?
> - A failed document lands as a row with `parse_status = 'failed'` and an error message. Nothing
>   retries it and nothing alerts anyone. Where should that responsibility live — in the procedure,
>   in a second task, or outside Snowflake entirely?


In [ ]:
%%sql -r pipeline_test_1
-- End-to-end test: upload a file and verify it flows through the pipeline
-- 1. Check current parsed document count
SELECT COUNT(*) AS parsed_so_far FROM GENAI_STUDY.PUBLIC.PARSED_DOCUMENTS;


In [ ]:
%%sql -r pipeline_test_2
-- 2. Check stream for pending files
SELECT METADATA$ACTION, RELATIVE_PATH, SIZE
FROM GENAI_STUDY.PUBLIC.DOCS_STAGE_STREAM
LIMIT 10;


In [ ]:
%%sql
-- 3. Manually trigger the pipeline
EXECUTE TASK GENAI_STUDY.PUBLIC.AUTO_PARSE_DOCUMENTS;


In [ ]:
%%sql -r pipeline_test_4
-- 4. Verify results landed
SELECT filename, invoice_number, invoice_date, total_amount, vendor_name, parse_status, parsed_at
FROM GENAI_STUDY.PUBLIC.PARSED_DOCUMENTS
ORDER BY parsed_at DESC
LIMIT 5;


> ### ⚠️ Common misconceptions
>
> **"`EXECUTE TASK` runs the whole graph, so I can test it that way."**
> `EXECUTE TASK` on the root triggers a graph run, but only the child tasks you have already
> resumed take part. An unresumed child is skipped in silence, so a manual test can pass while the
> second half of your pipeline has never executed once.
> → [Task graphs](https://docs.snowflake.com/en/user-guide/tasks-graphs)
>
> **"`TASK_HISTORY` will show me how the tasks are wired together."**
> `INFORMATION_SCHEMA.TASK_HISTORY()` reports runs — name, state, scheduled time, return value,
> error message. It has no predecessors column. Structure comes from `SHOW TASKS` or
> `INFORMATION_SCHEMA.TASK_DEPENDENTS`. Looking for the graph in the history view is how people
> conclude the DAG "isn't there".
> → [Task graphs](https://docs.snowflake.com/en/user-guide/tasks-graphs)
>
> **"A graph can be as wide as I need."**
> A task graph is limited to 1,000 tasks, and a single task to 100 predecessors and 100 children.
> Generating one task per document type hits that ceiling faster than it looks.
> → [Task graphs](https://docs.snowflake.com/en/user-guide/tasks-graphs)


---
## Scenario

**Situation.** A legal department uploads contracts to an S3 bucket behind an external stage. They
expect clause summaries within 30 minutes. Forty contracts went up this morning; none has been
processed, and nothing has errored.

**Question.** Give three debugging steps, in order.

### Worked solution

**1. Does the stream see anything?**

```sql
SELECT SYSTEM$STREAM_HAS_DATA('GENAI_STUDY.PUBLIC.DOCS_STAGE_STREAM') AS has_data;
SELECT COUNT(*) FROM GENAI_STUDY.PUBLIC.DOCS_STAGE_STREAM;
```

`FALSE` almost always means the directory table never refreshed — files written straight to S3 are
invisible to Snowflake until the metadata reconciles.

```sql
ALTER STAGE my_s3_stage REFRESH;
-- permanent fix: DIRECTORY = (ENABLE = TRUE AUTO_REFRESH = TRUE)
```

**2. Did the task run, and how did it end?**

```sql
SELECT NAME, STATE, SCHEDULED_TIME, ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
    SCHEDULED_TIME_RANGE_START => DATEADD('hour', -2, CURRENT_TIMESTAMP())));
```

`SKIPPED` means the `WHEN` condition was false, which sends you back to step 1. `FAILED` hands you
the error. No rows at all means the task never fired — step 3.

**3. Is the task actually resumed?**

```sql
SHOW TASKS LIKE 'AUTO_PARSE_DOCUMENTS' IN SCHEMA GENAI_STUDY.PUBLIC;
-- state = 'suspended'  ->  ALTER TASK ... RESUME;
```

Check the whole graph, not one task. Tasks are created suspended, and resuming the root does not
resume its children — `SYSTEM$TASK_DEPENDENTS_ENABLE` on the root does.

**Two more that produce the same silence**

4. **The task owner's privileges.** A task runs as its owner. It needs `EXECUTE TASK` on the
   account, `USAGE` on the warehouse, `USE AI FUNCTIONS` plus a Cortex database role, and stage
   access. A pipeline that works when you run it by hand and not on a schedule is this, nearly every
   time.
5. **Something else drained the stream.** A stream advances on any DML that reads it. If a manual
   `CALL` of the procedure ran at 09:00, the 09:05 task finds an empty stream and correctly does
   nothing. Two consumers need two streams.

**What this architecture gives up:** a stream tells you a file is new, not that it is good. Nothing
here reprocesses a document that failed, and the 30-minute promise is only as good as the slowest
link — which is usually the directory refresh, not the AI call.


---

## Check your understanding

Twelve questions on this notebook. Answer before expanding.

**1.** What state is a task in immediately after `CREATE TASK`?

<details><summary>Show answer</summary>

Suspended. Newly created and cloned tasks are created suspended, and stay that way until
`ALTER TASK … RESUME` or an explicit `EXECUTE TASK`. Nothing about the create statement warns you,
and the run history of a never-resumed task is empty rather than failing.

→ [CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)

</details>

**2.** Which stream type is supported on a directory table, and what are its metadata columns?

<details><summary>Show answer</summary>

Standard streams. The metadata columns are `METADATA$ACTION` (the recorded operation — `INSERT` or
`DELETE`), `METADATA$ISUPDATE` (whether it was part of an `UPDATE`), and `METADATA$ROW_ID` (a
unique, immutable row id). Filtering on `METADATA$ACTION = 'INSERT'` is how you process new files
and ignore removals.

→ [Introduction to streams](https://docs.snowflake.com/en/user-guide/streams-intro)

</details>

**3.** What exactly causes a stream to advance its offset?

<details><summary>Show answer</summary>

Being used in a DML transaction — an `INSERT … SELECT` from the stream, a CTAS, or a
`COPY INTO <location>`. A plain `SELECT` does not advance it. This is why you can inspect a stream
as often as you like while debugging, and why a stray CTAS in someone's worksheet can empty it.

→ [Introduction to streams](https://docs.snowflake.com/en/user-guide/streams-intro)

</details>

**4.** Forty files were uploaded to S3. `SYSTEM$STREAM_HAS_DATA` returns `FALSE`. What is wrong?

<details><summary>Show answer</summary>

The directory table has not been refreshed, so the stream has no change data to report. A stream on
a stage watches the directory table, and that metadata does not update itself for writes that
bypass Snowflake. Run `ALTER STAGE … REFRESH`, or configure `AUTO_REFRESH = TRUE` and the cloud
notifications behind it.

→ [CREATE STREAM](https://docs.snowflake.com/en/sql-reference/sql/create-stream)

</details>

**5.** A task's `WHEN SYSTEM$STREAM_HAS_DATA(...)` keeps firing every five minutes and the body
inserts nothing. What is happening?

<details><summary>Show answer</summary>

The body is not consuming the stream in a DML operation, so the offset never advances and the
function keeps returning `TRUE`. The documentation is explicit: when it returns `TRUE` you must
consume the stream in a DML operation. Note also that the function avoids false negatives but is not
guaranteed to avoid false positives, so an occasional empty run is expected — a permanent one is a
bug in the body.

→ [SYSTEM$STREAM_HAS_DATA](https://docs.snowflake.com/en/sql-reference/functions/system_stream_has_data)

</details>

**6.** A procedure calling `AI_EXTRACT` works when you run it and fails when the task runs it. Why?

<details><summary>Show answer</summary>

The task runs as its **owner**, and a stored procedure runs with owner's rights too — neither runs
as you. The owning role is missing something your role has: the `USE AI FUNCTIONS` account
privilege, one of the Cortex database roles, or `READ` on the stage. Granting the caller more
privileges changes nothing.

→ [CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)

</details>

**7.** You resume the root task of a two-task graph. The child never runs. Why?

<details><summary>Show answer</summary>

Resuming the root does not resume its children — each child is its own suspended object. Resume it
with `ALTER TASK … RESUME`, or call `SYSTEM$TASK_DEPENDENTS_ENABLE` on the root to enable the whole
graph at once. Before running a graph manually you must resume every child you want included.

→ [Task graphs](https://docs.snowflake.com/en/user-guide/tasks-graphs)

</details>

**8.** You need to see which task is a predecessor of which. Which object do you query?

<details><summary>Show answer</summary>

`SHOW TASKS`, whose output includes the predecessors, schedule, condition and state, or
`INFORMATION_SCHEMA.TASK_DEPENDENTS` with the root task, to walk the graph recursively.
`INFORMATION_SCHEMA.TASK_HISTORY()` is the tempting wrong answer: it reports runs — `STATE`,
`SCHEDULED_TIME`, `ERROR_MESSAGE` — not structure.

→ [Task graphs](https://docs.snowflake.com/en/user-guide/tasks-graphs)

</details>

**9.** The procedure in this notebook makes two AI calls per document. Justify that, then argue
against it.

<details><summary>Show answer</summary>

For: the table keeps `raw_text`, so later questions — a clause search, a re-extraction with better
wording, an audit — can be answered without touching the PDF again, and parsing is billed per page
whether you store the result or not. Against: it doubles the per-document AI spend for a column
nobody may read, and at 10,000 documents a day that is the largest line in the pipeline's bill. The
deciding question is whether your field list is stable; if it is, drop the parse.

→ [Extracting data from documents](https://docs.snowflake.com/en/user-guide/snowflake-cortex/document-extraction)

</details>

**10.** Serverless task or a task with `WAREHOUSE = COMPUTE_WH` for a document pipeline that runs
every five minutes but usually has nothing to do?

<details><summary>Show answer</summary>

Either can work, because the `WHEN` gate means an idle run starts no compute at all. A serverless
task sizes itself from `USER_TASK_MANAGED_INITIAL_WAREHOUSE_SIZE`, needs the `EXECUTE MANAGED TASK`
privilege, and removes the warehouse from your operational surface; a user-managed task lets you
share an existing warehouse and cap it deliberately — which matters here, since Snowflake recommends
nothing larger than MEDIUM for parsing. Bursty, unpredictable arrivals favour serverless; a
predictable batch on a warehouse you already run favours the explicit one.

→ [CREATE TASK](https://docs.snowflake.com/en/sql-reference/sql/create-task)

</details>

**11.** Two teams want to process the same uploaded documents for different purposes. What do you
build?

<details><summary>Show answer</summary>

Two streams on the same stage, one per consumer. A stream is a single bookmark: whichever consumer
performs DML first advances it, and the second finds nothing and is not told why. Sharing one
stream between two pipelines is a data-loss bug that only appears when both happen to run close
together.

→ [Introduction to streams](https://docs.snowflake.com/en/user-guide/streams-intro)

</details>

**12.** The pipeline lands `raw_text` for 4,000 contracts and legal now wants to ask questions
across all of them. What do you add, and what has to change about how the text is stored?

<details><summary>Show answer</summary>

A retrieval layer: chunk `raw_text` with
`SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(raw_text, 'markdown', <chunk_size>, <overlap>)` and
build a Cortex Search service over the chunk column. Snowflake recommends chunks of no more than 512
tokens — roughly 385 English words — so one row per document is too coarse; parsing with
`page_split` gives you a better starting granularity. Note also that `raw_text VARCHAR(16000)` will
truncate a long contract, so the storage shape has to change before the search layer can be honest.

→ [Cortex Search overview](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-search/cortex-search-overview)

</details>
